# exp016 back to feature

## 0. Experiment Metadata

In [1]:
# ========================================
# EXPERIMENT CONFIG
# ========================================

EXP_NAME = "exp016_back_to_feature"
TARGET = "Churn"
ID_COL = "id"

N_SPLITS = 5
SEED = 42

print(f"Running {EXP_NAME}")

Running exp016_back_to_feature


## 1. Imports

In [2]:
# ========================================
# IMPORTS
# ========================================

import os
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

import lightgbm as lgb
import xgboost as xgb

import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',50)

## 2. Reproducibility

In [3]:
# ========================================
# SEED EVERYTHING
# ========================================

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(SEED)

## 3. Load Data

In [4]:
# ========================================
# LOAD DATA
# ========================================

DATA_PATH = "/Users/theojeremiah/Workspace/01_DataScience/Projects/202603_Kaggle_CustomerChurn/data/raw/"

train = pd.read_csv(DATA_PATH + "train.csv")
test = pd.read_csv(DATA_PATH + "test.csv")

print(train.shape, test.shape)
train.head()

(594194, 21) (254655, 20)


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,No,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,Yes,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


## 4. Quick Sanity Checs (Fast EDA)

In [5]:
# ========================================
# QUICK EDA
# ========================================

print("Target distribution:")
print(train['Churn'].value_counts(normalize=True))

print("\nMissing values (top 10):")
print(train.isnull().mean().sort_values(ascending=False).head(10))

Target distribution:
Churn
No     0.774792
Yes    0.225208
Name: proportion, dtype: float64

Missing values (top 10):
id                  0.0
DeviceProtection    0.0
TotalCharges        0.0
MonthlyCharges      0.0
PaymentMethod       0.0
PaperlessBilling    0.0
Contract            0.0
StreamingMovies     0.0
StreamingTV         0.0
TechSupport         0.0
dtype: float64


## 5. Feature Engineering 

In [6]:
# ========================================
# FEATURE ENGINEERING (EXP007 - STRUCTURED SERVICE + BEST FEATURES)
# ========================================

internet_cols = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
]

def create_features(df):
    df = df.copy()

    # ========================================
    # CLEANING (VERY IMPORTANT)
    # ========================================
    # Convert "No internet service" → "No"
    for col in internet_cols:
        if col in df.columns:
            df[col] = df[col].replace("No internet service", "No")

    # Convert TotalCharges to numeric (common Kaggle issue)
    if "TotalCharges" in df.columns:
        df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

    # ========================================
    # FINANCIAL FEATURES (PROVEN STRONG)
    # ========================================
    if all(col in df.columns for col in ["TotalCharges", "tenure"]):
        df["avg_charge"] = df["TotalCharges"] / (df["tenure"] + 1)

    if all(col in df.columns for col in ["TotalCharges", "MonthlyCharges", "tenure"]):
        df["charge_diff"] = df["TotalCharges"] - (df["MonthlyCharges"] * df["tenure"])

    # ========================================
    # PHONE SERVICE BUNDLE
    # ========================================
    if all(col in df.columns for col in ["PhoneService", "MultipleLines"]):
        df["has_phone"] = (df["PhoneService"] == "Yes").astype(int)
        df["multi_line"] = (df["MultipleLines"] == "Yes").astype(int)
        df["phone_complexity"] = df["has_phone"] + df["multi_line"]

    # ========================================
    # INTERNET SERVICE BUNDLE (VERY STRONG)
    # ========================================
    if "InternetService" in df.columns:
        df["has_internet"] = (df["InternetService"] != "No").astype(int)

    if all(col in df.columns for col in internet_cols):
        df["internet_services"] = (df[internet_cols] == "Yes").sum(axis=1)

    if "has_internet" in df.columns and "internet_services" in df.columns:
        df["internet_complexity"] = df["has_internet"] + df["internet_services"]

    # ========================================
    # TOTAL SERVICE POWER (HIGH SIGNAL)
    # ========================================
    if "phone_complexity" in df.columns and "internet_complexity" in df.columns:
        df["total_services"] = df["phone_complexity"] + df["internet_complexity"]

    # ========================================
    # CUSTOMER TYPE (KAGGLE GOLD FEATURE)
    # ========================================
    if all(col in df.columns for col in ["PhoneService", "InternetService"]):
        df["customer_type"] = (
            df["PhoneService"].astype(str) + "_" +
            df["InternetService"].astype(str)
        )

    # ========================================
    # BEHAVIORAL RATIOS (VERY IMPORTANT)
    # ========================================
    if all(col in df.columns for col in ["total_services", "tenure"]):
        df["services_per_tenure"] = df["total_services"] / (df["tenure"] + 1)

    if all(col in df.columns for col in ["MonthlyCharges", "total_services"]):
        df["charges_per_service"] = df["MonthlyCharges"] / (df["total_services"] + 1)

    # ========================================
    # BUSINESS LOGIC FEATURES
    # ========================================
    if "PaymentMethod" in df.columns:
        df["payment_risk"] = (df["PaymentMethod"] == "Electronic check").astype(int)
        df["is_auto_pay"] = df["PaymentMethod"].isin([
            "Bank transfer (automatic)",
            "Credit card (automatic)"
        ]).astype(int)

    if "InternetService" in df.columns:
        df["is_fiber"] = (df["InternetService"] == "Fiber optic").astype(int)
        df["no_internet"] = (df["InternetService"] == "No").astype(int)

    # ========================================
    # TENURE SIGNAL (STILL IMPORTANT)
    # ========================================
    if "tenure" in df.columns:
        df["tenure_log"] = np.log1p(df["tenure"])
        df["is_new"] = (df["tenure"] < 6).astype(int)

    return df


# ========================================
# APPLY
# ========================================
train = create_features(train)
test = create_features(test)

### Feature Testing

In [9]:
# ========================================
# CONFIG
# ========================================
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import numpy as np
import pandas as pd

SEED = 42
N_SPLITS = 3

# ========================================
# BASELINE MODEL
# ========================================
def run_cv(X, y):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof = np.zeros(len(X))

    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = lgb.LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            random_state=SEED,
            n_jobs=-1
        )

        model.fit(X_tr, y_tr)
        oof[val_idx] = model.predict_proba(X_val)[:, 1]

    return roc_auc_score(y, oof)


# ========================================
# FEATURE FUNCTIONS
# ========================================
def feat_avg_charge(df):
    df["avg_charge"] = df["TotalCharges"] / (df["tenure"] + 1)
    return df

def feat_num_services(df):
    service_cols = [
        "OnlineSecurity", "OnlineBackup", "DeviceProtection", 
        "TechSupport", "StreamingTV", "StreamingMovies"
    ]
    df["num_services"] = (df[service_cols] == "Yes").sum(axis=1)
    return df

def feat_risk_score(df):
    df["risk_score"] = (
        (df["Contract"] == "Month-to-month").astype(int) +
        (df["PaymentMethod"] == "Electronic check").astype(int) +
        (df["InternetService"] == "Fiber optic").astype(int)
    )
    return df

feature_functions = {
    "avg_charge": feat_avg_charge,
    "num_services": feat_num_services,
    "risk_score": feat_risk_score,
}


# ========================================
# ENCODING
# ========================================
def encode_data(train_df, test_df):
    full = pd.concat([train_df, test_df], axis=0)

    for col in full.columns:
        if full[col].dtype == "object":
            full[col] = full[col].fillna("MISSING")
            full[col] = full[col].astype("category").cat.codes

    train_enc = full.iloc[:len(train_df)].copy()
    test_enc = full.iloc[len(train_df):].copy()

    return train_enc, test_enc


# ========================================
# BASELINE
# ========================================
train_base = train.copy()
test_base = test.copy()

train_enc, test_enc = encode_data(train_base, test_base)

X_base = train_enc.drop(columns=[TARGET])
y = train_enc[TARGET]

baseline_score = run_cv(X_base, y)
print(f"\n🔥 Baseline AUC: {baseline_score:.5f}")


# ========================================
# 🔥 STEP 1: TEST DROPPING ORIGINAL FEATURES
# ========================================
original_cols = ["TotalCharges", "MonthlyCharges", "tenure"]

best_score = baseline_score
best_config = X_base.columns.tolist()

print("\n==============================")
print("TEST DROP ORIGINAL FEATURES")
print("==============================")

for col in original_cols:
    if col in X_base.columns:
        X_drop = X_base.drop(columns=[col])
        score = run_cv(X_drop, y)

        print(f"Drop {col} → AUC: {score:.5f} | Gain: {score - best_score:.6f}")

        if score > best_score:
            best_score = score
            best_config = X_drop.columns.tolist()
            print(f"🔥 ACCEPT DROP: {col}")


# ========================================
# 🔥 STEP 2: FEATURE + DROP COMBINATIONS
# ========================================
results = []

for name, fn in feature_functions.items():

    print(f"\n==============================")
    print(f"Testing Feature: {name}")
    print(f"==============================")

    # apply feature
    train_tmp = fn(train.copy())
    test_tmp = fn(test.copy())

    train_enc, test_enc = encode_data(train_tmp, test_tmp)
    X = train_enc.drop(columns=[TARGET])

    # ---------- BASE ADD ----------
    score_add = run_cv(X, y)
    best_local_score = score_add
    best_local_config = X.columns.tolist()

    print(f"ADD ONLY → AUC: {score_add:.5f}")

    # ---------- ADD + DROP ----------
    for col in original_cols:
        if col in X.columns:
            X_drop = X.drop(columns=[col])
            score = run_cv(X_drop, y)

            print(f"ADD + DROP {col} → AUC: {score:.5f}")

            if score > best_local_score:
                best_local_score = score
                best_local_config = X_drop.columns.tolist()

    gain = best_local_score - best_score

    print(f"FINAL → AUC: {best_local_score:.5f} | Gain: {gain:.6f}")

    keep = False
    if gain > 0:
        keep = True
        best_score = best_local_score
        best_config = best_local_config
        print(f"✅ KEEP FEATURE: {name}")
    else:
        print(f"❌ DROP FEATURE: {name}")

    results.append({
        "feature": name,
        "score": best_local_score,
        "gain": gain,
        "kept": keep
    })


# ========================================
# RESULT
# ========================================
results_df = pd.DataFrame(results).sort_values(by="gain", ascending=False)

print("\n🏆 FEATURE RESULT")
print(results_df)

print("\n🔥 FINAL FEATURES")
print(best_config)

print(f"\n🚀 FINAL CV: {best_score:.5f}")

[LightGBM] [Info] Number of positive: 89211, number of negative: 306918
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003438 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 881
[LightGBM] [Info] Number of data points in the train set: 396129, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225207 -> initscore=-1.235576
[LightGBM] [Info] Start training from score -1.235576
[LightGBM] [Info] Number of positive: 89211, number of negative: 306918
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003940 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 881
[LightGBM] [Info] Number of data points in the train set: 396129, number of used features: 20
[LightGBM] [Info

In [7]:
results_df

,feature,score,gain,drop_gain
0,avg_charge,0.915327,0.000189,-0.000591
1,charge_diff,0.915236,0.000098,-0.000878
3,risk_score,0.915202,0.000064,NaN
4,fiber_monthly,0.915193,0.000055,NaN
2,num_services,0.915148,0.000010,NaN
5,service_density,0.915144,0.000006,NaN


## 6. Target Encoding

In [7]:
# ========================================
# TARGET ENCODING (SIMPLE VERSION)
# ========================================

from category_encoders import TargetEncoder

# Get categorical columns from TRAIN
categorical_cols = train.select_dtypes(include="object").columns.tolist()

# Remove target column
categorical_cols = [col for col in categorical_cols if col != TARGET]

# Initialize encoder
encoder = TargetEncoder(cols=categorical_cols)

# Fit on train, transform both
train[categorical_cols] = encoder.fit_transform(train[categorical_cols], train[TARGET])
test[categorical_cols] = encoder.transform(test[categorical_cols])

## 7. Prepare Data

In [8]:
# ========================================
# PREPARE DATA
# ========================================

features = [col for col in train.columns if col not in [TARGET, ID_COL]]

train[TARGET] = train[TARGET].map({"No": 0, "Yes": 1}).astype(int)

X = train[features]
y = train[TARGET]

X_test = test[features]

## 8. Validation Strategy

In [9]:
# ========================================
# CV SETUP
# ========================================

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

## 9. Model Training

In [10]:
# ========================================
# SEED EVERYTHING
# ========================================
def seed_everything(seed):
    import random, os
    import numpy as np

    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


# ========================================
# CONFIG
# ========================================
SEEDS = [42, 52, 62]
N_SPLITS = 5

oof_preds_lgb = np.zeros(len(train))
oof_preds_xgb = np.zeros(len(train))

test_preds_lgb = np.zeros(len(test))
test_preds_xgb = np.zeros(len(test))


# ========================================
# MULTI-SEED TRAINING
# ========================================
for seed in SEEDS:
    
    print(f"\n====================")
    print(f"SEED {seed}")
    print(f"====================")

    seed_everything(seed)

    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=seed
    )

    oof_lgb_seed = np.zeros(len(train))
    oof_xgb_seed = np.zeros(len(train))

    test_lgb_seed = np.zeros(len(test))
    test_xgb_seed = np.zeros(len(test))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        
        print(f"\n--- Fold {fold} ---")

        X_train, X_valid = X.iloc[tr_idx], X.iloc[val_idx]
        y_train, y_valid = y.iloc[tr_idx], y.iloc[val_idx]

        # ======================
        # LIGHTGBM
        # ======================
        lgb_model = lgb.LGBMClassifier(
            n_estimators=5000,
            learning_rate=0.01,
            num_leaves=64,
            max_depth=-1,
            min_child_samples=30,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.5,
            reg_lambda=0.5,
            random_state=seed,
            n_jobs=-1
        )

        lgb_model.fit(
            X_train, y_train,
            eval_set=[(X_valid, y_valid)],
            callbacks=[
                lgb.early_stopping(100),
                lgb.log_evaluation(200)
            ]
        )

        val_pred_lgb = lgb_model.predict_proba(X_valid)[:, 1]
        oof_lgb_seed[val_idx] = val_pred_lgb
        test_lgb_seed += lgb_model.predict_proba(X_test)[:, 1] / N_SPLITS


        # ======================
        # XGBOOST
        # ======================
        xgb_model = xgb.XGBClassifier(
            n_estimators=5000,
            learning_rate=0.01,
            max_depth=4,
            min_child_weight=3,
            subsample=0.8,
            colsample_bytree=0.8,
            gamma=1,
            reg_alpha=0.5,
            reg_lambda=1.0,
            eval_metric="auc",
            random_state=seed,
            n_jobs=-1,
            tree_method="hist"
        )

        xgb_model.fit(
            X_train, y_train,
            eval_set=[(X_valid, y_valid)],
            verbose=False
        )

        val_pred_xgb = xgb_model.predict_proba(X_valid)[:, 1]
        oof_xgb_seed[val_idx] = val_pred_xgb
        test_xgb_seed += xgb_model.predict_proba(X_test)[:, 1] / N_SPLITS


    # ======================
    # ACCUMULATE RESULTS
    # ======================
    oof_preds_lgb += oof_lgb_seed / len(SEEDS)
    oof_preds_xgb += oof_xgb_seed / len(SEEDS)

    test_preds_lgb += test_lgb_seed / len(SEEDS)
    test_preds_xgb += test_xgb_seed / len(SEEDS)


# ========================================
# CV SCORES (INDIVIDUAL)
# ========================================
score_lgb = roc_auc_score(y, oof_preds_lgb)
score_xgb = roc_auc_score(y, oof_preds_xgb)

print("\nLGB CV:", score_lgb)
print("XGB CV:", score_xgb)


# ========================================
# ENSEMBLE WEIGHT OPTIMIZATION
# ========================================
best_score = 0
best_w = 0

for w in np.arange(0.0, 1.01, 0.01):
    oof_comb = w * oof_preds_lgb + (1 - w) * oof_preds_xgb
    score = roc_auc_score(y, oof_comb)

    if score > best_score:
        best_score = score
        best_w = w

print(f"\nBest Weight (LGB): {best_w:.2f}")
print(f"Best Ensemble CV: {best_score:.5f}")


# ========================================
# FINAL TEST PREDICTIONS
# ========================================
test_preds = best_w * test_preds_lgb + (1 - best_w) * test_preds_xgb


SEED 42

--- Fold 0 ---
[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011527 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1754
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 37
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225209 -> initscore=-1.235567
[LightGBM] [Info] Start training from score -1.235567
Training until validation scores don't improve for 100 rounds
[200]	valid_0's binary_logloss: 0.316803
[400]	valid_0's binary_logloss: 0.302368
[600]	valid_0's binary_logloss: 0.300244
[800]	valid_0's binary_logloss: 0.299512
[1000]	valid_0's binary_logloss: 0.299091
[1200]	valid_0's binary_logloss: 0.298855
[1400]	valid_0's binary_logloss: 0.298694
[1600]	valid_0's binary_logloss: 0.298581
[1800]	valid_0's bin

#### prev score 0.91669

## 11. Feature Importance

In [113]:
# ========================================
# FEATURE IMPORTANCE
# ========================================

import matplotlib.pyplot as plt

importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values(by="importance", ascending=False)

importance.head(20)

plt.figure(figsize=(8, 10))
plt.barh(importance["feature"].head(20), importance["importance"].head(20))
plt.gca().invert_yaxis()
plt.title("Top Features")
plt.show()

ValueError: All arrays must be of the same length

## 12. Create Submission

In [11]:
# ========================================
# SUBMISSION (PROBABILITY - CORRECT)
# ========================================

import os

# Define path
OUTPUT_DIR = "/Users/theojeremiah/Workspace/01_DataScience/Projects/202603_Kaggle_CustomerChurn/outputs/submissions/"

# Create folder if not exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Create submission (USE PROBABILITY)
submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET: test_preds   # ✅ probability, NOT binary
})

# File path
file_path = os.path.join(OUTPUT_DIR, f"{EXP_NAME}.csv")

# Save
submission.to_csv(file_path, index=False)

print(f"Submission saved at: {file_path}")

Submission saved at: /Users/theojeremiah/Workspace/01_DataScience/Projects/202603_Kaggle_CustomerChurn/outputs/submissions/exp015_exp14feat_exp9model.csv
